# 🚀 FLARE — Huấn luyện lại YOLO11n trên dữ liệu phản hồi (Continual Learning)
> Notebook này được hệ thống FLARE Web tự động sinh và kích hoạt trên Kaggle GPU.
> **Mục tiêu:** Fine-tune mô hình phát hiện Khói/Lửa bằng cách trộn dữ liệu phản hồi đã duyệt (Hard Negatives + Hard Positives) với tập huấn luyện gốc.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q ultralytics mlflow
import os, sys, shutil, zipfile, io, base64, time, glob
from pathlib import Path
import torch
from ultralytics import YOLO
import mlflow

print('GPU sẵn sàng:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Thiết bị GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Giải nén tập dữ liệu Feedback được Web đóng gói tự động
FEEDBACK_B64 = "..."
WORK_DIR = Path('/kaggle/working')
FEEDBACK_DIR = WORK_DIR / 'feedback_dataset'
FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)

zip_data = base64.b64decode(FEEDBACK_B64)
with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    z.extractall(FEEDBACK_DIR)

fb_images = list((FEEDBACK_DIR / 'images').glob('*.*'))
fb_labels = list((FEEDBACK_DIR / 'labels').glob('*.txt'))
print(f'✅ Đã giải nén thành công {len(fb_images)} ảnh và {len(fb_labels)} file nhãn feedback!')


In [ ]:
# 3. Tải checkpoint Baseline Model từ DagsHub MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "nguyenthanhquyen145"
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "<YOUR_DAGSHUB_TOKEN>")
mlflow.set_tracking_uri("https://dagshub.com/nguyenthanhquyen145/Real_Time_Smoke_Fire_Detection_YOLO.mlflow")

print('Đang tải checkpoint Baseline Model từ DagsHub MLflow...')
baseline_pt = WORK_DIR / 'baseline_best.pt'
try:
    # Tải artifact từ run gốc
    downloaded = mlflow.artifacts.download_artifacts(run_id="fafc4786b6f24e2f851b1164ac3574f2", artifact_path="model/best.pt", dst_path=str(WORK_DIR))
    if Path(downloaded).is_file():
        shutil.copy2(downloaded, baseline_pt)
        print('✅ Tải baseline weights thành công từ DagsHub!')
except Exception as e:
    print('⚠️ Tải từ MLflow gặp lỗi, chuyển sang nạp weights mặc định:', e)
    # Fallback dự phòng nếu có checkpoint cục bộ hoặc tải yolo11n gốc
    baseline_pt = 'yolo11n.pt'


In [ ]:
# 4. Xác định tập dữ liệu huấn luyện gốc (Base Dataset)
BASE_CANDIDATES = [
    Path('/kaggle/input/finalsfd-yolo-dataset/FinalSFD_YOLO_Dataset'),
    Path('/kaggle/input/datasets/quynnguynthanh/finalsfd-yolo-dataset/FinalSFD_YOLO_Dataset'),
    Path('/kaggle/input/finalsfd-yolo-dataset'),
]
BASE_DIR = None
for cand in BASE_CANDIDATES:
    if cand.is_dir() and (cand / 'images' / 'train').is_dir():
        BASE_DIR = cand
        break

if not BASE_DIR:
    print('⚠️ Không tìm thấy FinalSFD_YOLO_Dataset, tìm kiếm các thư mục trong /kaggle/input:')
    print(list(Path('/kaggle/input').glob('*')))
    # Fallback: sử dụng trực tiếp thư mục feedback nếu dataset gốc chưa mount
    BASE_DIR = FEEDBACK_DIR
else:
    print(f'✅ Đã phát hiện Base Dataset tại: {BASE_DIR}')


In [ ]:
# 5. Trộn dữ liệu (Data Mixing / Experience Replay)
MERGED_DIR = WORK_DIR / 'dataset_retrain'
MERGED_TRAIN_IMG = MERGED_DIR / 'images' / 'train'
MERGED_TRAIN_LBL = MERGED_DIR / 'labels' / 'train'
MERGED_VAL_IMG = MERGED_DIR / 'images' / 'val'
MERGED_VAL_LBL = MERGED_DIR / 'labels' / 'val'

MERGED_TRAIN_IMG.mkdir(parents=True, exist_ok=True)
MERGED_TRAIN_LBL.mkdir(parents=True, exist_ok=True)
MERGED_VAL_IMG.mkdir(parents=True, exist_ok=True)
MERGED_VAL_LBL.mkdir(parents=True, exist_ok=True)

# Tạo symlink hoặc copy từ Base Dataset để tiết kiệm dung lượng đĩa
if BASE_DIR != FEEDBACK_DIR:
    print('Đang liên kết dữ liệu huấn luyện gốc...')
    for p in (BASE_DIR / 'images' / 'train').glob('*.*'):
        dst = MERGED_TRAIN_IMG / p.name
        if not dst.exists(): os.symlink(p, dst)
    for p in (BASE_DIR / 'labels' / 'train').glob('*.txt'):
        dst = MERGED_TRAIN_LBL / p.name
        if not dst.exists(): os.symlink(p, dst)
    for p in (BASE_DIR / 'images' / 'val').glob('*.*'):
        dst = MERGED_VAL_IMG / p.name
        if not dst.exists(): os.symlink(p, dst)
    for p in (BASE_DIR / 'labels' / 'val').glob('*.txt'):
        dst = MERGED_VAL_LBL / p.name
        if not dst.exists(): os.symlink(p, dst)

# Đưa toàn bộ các mẫu feedback vào tập Train
print('Đang hòa trộn các mẫu feedback đã kiểm duyệt vào tập Train...')
for p in (FEEDBACK_DIR / 'images').glob('*.*'):
    shutil.copy2(p, MERGED_TRAIN_IMG / f'fb_{p.name}')
for p in (FEEDBACK_DIR / 'labels').glob('*.txt'):
    shutil.copy2(p, MERGED_TRAIN_LBL / f'fb_{p.name}')

# Tạo file retrain_data.yaml chuẩn YOLO
yaml_content = f'''# Cấu hình tự động sinh cho Fine-tuning FLARE
path: {MERGED_DIR.as_posix()}
train: images/train
val: images/val
test: images/val
names:
  0: smoke
  1: fire
'''
DATA_YAML = WORK_DIR / 'retrain_data.yaml'
DATA_YAML.write_text(yaml_content, encoding='utf-8')
print(f'✅ Hoàn tất hòa trộn dữ liệu! Tổng ảnh train: {len(list(MERGED_TRAIN_IMG.glob("*.*")))}')


In [ ]:
# 6. Khởi chạy Fine-tuning với YOLO11 (Continual Learning)
print('🚀 Bắt đầu quá trình Fine-tuning...')
model = YOLO(str(baseline_pt))

# Siêu tham số tối ưu cho fine-tuning: lr0 thấp, freeze 10 layers đầu, 25 epochs
EPOCHS = 25
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=32,
    lr0=0.002,
    lrf=0.01,
    freeze=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/kaggle/working/retrain_run',
    name='exp',
    exist_ok=True,
    save=True,
    plots=True,
)
print('✅ Huấn luyện hoàn tất!')


In [ ]:
# 7. Đánh giá và Tự động đẩy kết quả lên DagsHub MLflow
best_weights = Path('/kaggle/working/retrain_run/exp/weights/best.pt')
if not best_weights.is_file():
    best_weights = Path('/kaggle/working/retrain_run/exp/weights/last.pt')

print('Đang đánh giá mô hình mới trên tập val...')
eval_model = YOLO(str(best_weights))
val_metrics = eval_model.val(data=str(DATA_YAML), split='val', imgsz=640)

timestamp = time.strftime('%Y%m%d_%H%M%S')
run_name = f'retrain_yolo11n_{timestamp}'
print(f'Đang đẩy mô hình {{run_name}} lên DagsHub MLflow...')

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({
        'epochs': EPOCHS,
        'imgsz': 640,
        'batch': 32,
        'lr0': 0.002,
        'freeze': 10,
        'base_model': 'final_sfd_yolo11n',
        'feedback_count': len(fb_images),
    })
    mlflow.log_metrics({
        'mAP50': float(val_metrics.box.map50),
        'mAP50-95': float(val_metrics.box.map),
        'precision': float(val_metrics.box.mp),
        'recall': float(val_metrics.box.mr),
    })
    if best_weights.is_file():
        mlflow.log_artifact(str(best_weights), artifact_path='model')
        print('✅ Đã đẩy file trọng số weights/best.pt lên MLflow!')

print('🎉 TOÀN BỘ CHU TRÌNH HUẤN LUYỆN LẠI ĐÃ HOÀN TẤT THÀNH CÔNG!')
